In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time

# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")
# Apply StandardScaler
scaler = StandardScaler()
X_sp = scaler.fit_transform(X)

print("Scaled features shape:", X_sp.shape)

Scaled features shape: (2930, 172)


In [2]:
#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_sp)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_pca.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (2930, 25)
Explained variance ratio sum: 0.9587142612380308


In [3]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [4]:
#K-Means on Scaled + PCA Data
start_time = time.time()
kmean_pca = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    kmean_pca.append({"algorithm": "KMeans", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
    
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 6.314162015914917 seconds
K-Means runtime: 6.3142 seconds


In [5]:
#Gaussian Mixture (GMM)on Scaled + PCA Data
start_time = time.time()
gmm_pca = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    gmm_pca.append({"algorithm": "GMM", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")   

Runtime: 32.7490348815918 seconds
GMM runtime: 32.7490 seconds


In [6]:
#Agglomerative Clustering on Scaled + PCA Data
start_time = time.time()
agg_pca = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")    

Runtime: 5.88076376914978 seconds
Agglomerative runtime: 5.8808 seconds


In [7]:
#Spectral Clustering on Scaled + PCA Data
start_time = time.time()
spec_pca = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    spec_pca.append({"algorithm": "Spectral", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")   

Runtime: 11.90206789970398 seconds
Spectral runtime: 11.9021 seconds


In [8]:
#DBSCAN on Scaled + PCA Data
start_time = time.time()
dbscan_pca = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_pca[mask], labels[mask])
        dbscan_pca.append({"algorithm": "DBSCAN", "preprocessing": "PCA", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 0.13730072975158691 seconds
DBSCAN runtime: 0.1373 seconds


In [9]:
from sklearn.cluster import OPTICS
start_time = time.time()

optics_pca = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_pca)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_pca, labels)
        optics_pca.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


Runtime: 32.333680629730225 seconds
Optics runtime: 32.3337 seconds


In [10]:
from sklearn.cluster import Birch
start_time = time.time()

birch_pca = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_pca)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_pca, labels)
        birch_pca.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Runtime: 11.614181280136108 seconds
Birch runtime: 11.6142 seconds


In [ ]:
import csv

ames_results_pca = (kmean_pca + gmm_pca + agg_pca + spec_pca + dbscan_pca+birch_pca+optics_pca)

# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/ames_data/ames_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_pca)


In [11]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

# ARI stability analysis 
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmean_pca:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_pca:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_pca:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spec_pca:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_pca:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_pca:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_pca:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# --- helper function to fit a model and return labels ---
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_pca)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_pca), size=len(X_pca), replace=True)
        X_boot = X_pca[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\n---- BOOTSTRAP ARI STABILITY ----")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\S


---- BOOTSTRAP ARI STABILITY ----
    algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples
      K-Means 2.0    0.9889   0.0084  NaN        NaN          NaN
      K-Means 3.0    0.9269   0.1722  NaN        NaN          NaN
      K-Means 4.0    0.9635   0.0282  NaN        NaN          NaN
      K-Means 5.0    0.8447   0.1423  NaN        NaN          NaN
      K-Means 6.0    0.7677   0.1005  NaN        NaN          NaN
      K-Means 7.0    0.6959   0.0993  NaN        NaN          NaN
      K-Means 8.0    0.7279   0.1030  NaN        NaN          NaN
          GMM 2.0    0.9889   0.0097  NaN        NaN          NaN
          GMM 3.0    0.8337   0.2127  NaN        NaN          NaN
          GMM 4.0    0.5298   0.1602  NaN        NaN          NaN
          GMM 5.0    0.3990   0.1023  NaN        NaN          NaN
          GMM 6.0    0.4955   0.1067  NaN        NaN          NaN
          GMM 7.0    0.4448   0.0981  NaN        NaN          NaN
          GMM 8.0    0.4512   0.1018  NaN

In [12]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  Stability Score
   DBSCAN NaN    1.0000   0.0000           1.0000
    BIRCH NaN    0.9986   0.0019           0.9962
    BIRCH NaN    0.9940   0.0039           0.9922


In [13]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples  Stability Score
   DBSCAN NaN    1.0000   0.0000  1.0        NaN          NaN           1.0000
    BIRCH NaN    0.9986   0.0019  NaN        0.2          NaN           0.9962
    BIRCH NaN    0.9940   0.0039  NaN        0.5          NaN           0.9922


In [14]:
ari_df.to_csv("updated_data/ARI_Score/ames_pca_ari.csv", index=False)

In [14]:
# Combine all algorithm results
all_results = (
    kmean_pca +
    gmm_pca +
    agg_pca +
    spec_pca +
    dbscan_pca +
    birch_pca +
    optics_pca
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#  Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\nTOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\nTOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better) 
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

#  Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\nBOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

# Bottom 3 by Davies-Bouldin (higher is worse)
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\nBOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

#  Bottom 3 by Calinski-Harabasz (lower is worse) 
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


TOP 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
   DBSCAN NaN  1.0        NaN          NaN         NaN      0.9078
   DBSCAN NaN  1.5        NaN          NaN         NaN      0.5328
   KMeans 2.0  NaN        NaN          NaN         NaN      0.1997

TOP 3 DAVIES-BOULDIN 
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
    BIRCH NaN  NaN        0.2          NaN      2919.0          0.0276
    BIRCH NaN  NaN        0.5          NaN      2893.0          0.0891
   DBSCAN NaN  1.0        NaN          NaN         NaN          0.1180

 TOP 3 CALINSKI-HARABASZ 
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
    BIRCH NaN  NaN        0.2          NaN      2919.0          1276.3013
   KMeans 2.0  NaN        NaN          NaN         NaN           770.7767
 Spectral 2.0  NaN        NaN          NaN         NaN           746.4423

BOTTOM 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  silho

In [15]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmean_pca,
    "GMM": gmm_pca,
    "Agglomerative": agg_pca,
    "Spectral": spec_pca,
    "DBSCAN": dbscan_pca,
    "BIRCH": birch_pca,
    "OPTICS": optics_pca
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
   
    print(algorithm)
   




    # Select parameter column


    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break



   
    # TOP 3 SILHOUETTE
  

    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )




    # TOP 3 DAVIES-BOULDIN
   

    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



  
    # TOP 3 CALINSKI-HARABASZ


    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1997
 3      0.1707
 4      0.1235

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.8930
 3          2.0969
 8          2.2860

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2           770.7767
 3           525.7886
 4           441.4108


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1857
 3      0.1448
 4      0.0791

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.9423
 3          2.0195
 6          2.8223

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2           679.9824
 3           430.0776
 4           321.3805


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1837
 3      0.1159
 4      0.1065

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.9604
 5          2.3478
 6          2.3949

Top 3 Calinski-H